# Regime-Aware Cross-Asset Market Risk

This notebook studies how a simple cross-asset portfolio behaves when volatility, correlation and common-factor concentration change through time. The emphasis is on walk-forward risk measurement and model validation rather than on forecasting returns.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import pandas as pd

from market_risk.backtesting import kupiec_pof_test, rolling_ewma_var, rolling_historical_var, var_exceptions
from market_risk.config import DEFAULT_REGIME_WINDOW, DEFAULT_START_DATE, DEFAULT_VAR_WINDOW, DEFAULT_WEIGHTS
from market_risk.data import download_adjusted_close, simple_returns
from market_risk.portfolio import component_volatility_contributions, portfolio_returns
from market_risk.regimes import expanding_percentile_score, label_regimes, rolling_regime_features
from market_risk.risk_metrics import historical_expected_shortfall, historical_var, parametric_var
from market_risk.stress_testing import historical_worst_window, reverse_stress_test


## 1. Data and portfolio construction

In [ ]:
tickers = list(DEFAULT_WEIGHTS)
prices = download_adjusted_close(tickers, start=DEFAULT_START_DATE)
asset_returns = simple_returns(prices)
port_ret = portfolio_returns(asset_returns, DEFAULT_WEIGHTS)
port_ret.describe()

## 2. Baseline risk measures

In [ ]:
risk_summary = pd.DataFrame({
    '95%': {
        'Historical VaR': historical_var(port_ret, 0.95),
        'Parametric VaR': parametric_var(port_ret, 0.95),
        'Historical ES': historical_expected_shortfall(port_ret, 0.95),
    },
    '99%': {
        'Historical VaR': historical_var(port_ret, 0.99),
        'Parametric VaR': parametric_var(port_ret, 0.99),
        'Historical ES': historical_expected_shortfall(port_ret, 0.99),
    },
})
risk_summary

## 3. Regime indicators

In [ ]:
features = rolling_regime_features(asset_returns, port_ret, window=DEFAULT_REGIME_WINDOW)
stress_score = expanding_percentile_score(features)
regimes = label_regimes(stress_score)
regime_frame = features.join(stress_score).join(regimes)
regime_frame.tail()

In [ ]:
ax = regime_frame['regime_stress_score'].plot(figsize=(11, 4), title='Regime stress score')
ax.set_ylabel('Expanding percentile score')
plt.show()

## 4. Walk-forward VaR backtesting

In [ ]:
hist_var_99 = rolling_historical_var(port_ret, window=DEFAULT_VAR_WINDOW, confidence=0.99)
ewma_var_99 = rolling_ewma_var(port_ret, confidence=0.99)
hist_exceptions = var_exceptions(port_ret, hist_var_99)
ewma_exceptions = var_exceptions(port_ret, ewma_var_99)
pd.DataFrame({
    'Historical VaR': kupiec_pof_test(hist_exceptions, 0.99),
    'EWMA VaR': kupiec_pof_test(ewma_exceptions, 0.99),
})

## 5. Risk attribution and reverse stress testing

In [ ]:
recent_cov = asset_returns.tail(DEFAULT_VAR_WINDOW).cov()
contributions = component_volatility_contributions(recent_cov, DEFAULT_WEIGHTS)
reverse_shock = reverse_stress_test(recent_cov, DEFAULT_WEIGHTS, target_loss=0.05)
display(contributions)
display(reverse_shock.to_frame())
historical_worst_window(port_ret, horizon=5)

## 6. Interpretation

The next step is empirical: compare VaR breaches across the regime score, inspect whether correlation and PC1 concentration rise around major drawdowns, and test sensitivity to the rolling-window and EWMA choices. No numerical claim should be added to the README or CV until these checks have been run and reproduced.